# 02 — Análisis Descriptivo (Parte 1)
**IA Generativa y Saber Pro 2021-2024** | Eduardo A. Victoria Cadena

Compara 2021-2022 vs 2023-2024. Reporta medias, DEs, pruebas t de Welch, Mann-Whitney, χ², y corrección por pruebas múltiples (Holm / Benjamini-Hochberg).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_PROYECTO = '/content/drive/MyDrive/IA-Y-EDUCACION-SUPERIOR'
import sys; sys.path.insert(0, RUTA_PROYECTO + '/python')

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings; warnings.filterwarnings('ignore')
from utils import (MODULOS_GENERICOS, MODULOS_ETIQUETAS, ETIQUETAS_VARS,
                   ALPHA, guardar_tabla, CONTROLES, DUMMIES_DEPTO)

PROC_DIR  = f'{RUTA_PROYECTO}/data/processed'
TBL_DIR   = f'{RUTA_PROYECTO}/outputs/tablas'
df = pd.read_pickle(f'{PROC_DIR}/datos_analisis.pkl')
print(f"Datos cargados: {len(df):,} filas")

In [ ]:
# ── Tabla resumen univariado ──────────────────────────────────────────────
vars_num = ['puntaje_saberpro_generico'] + MODULOS_GENERICOS + [
    'puntaje_saber11','estrato','nivel_educ_padre','distancia_bogota_km']
vars_num = [v for v in vars_num if v in df.columns]

resumen = (df[vars_num]
           .describe(percentiles=[0.25, 0.5, 0.75])
           .T
           .rename(columns={'count':'n','50%':'mediana','25%':'p25','75%':'p75'}))
resumen['pct_na'] = df[vars_num].isna().mean().mul(100).round(1)
resumen = resumen[['n','mean','std','min','p25','mediana','p75','max','pct_na']].round(3)
guardar_tabla(resumen.reset_index(names='variable'), 'T0_resumen_univariado', TBL_DIR)
resumen

In [ ]:
# ── Tabla 3: Comparación descriptiva por período ─────────────────────────
def comparar_periodos(df):
    rows = []
    d0 = df[df['periodo_ia']==0]
    d1 = df[df['periodo_ia']==1]

    # Variables continuas
    vars_cont = ['puntaje_saberpro_generico'] + MODULOS_GENERICOS + [
                 'puntaje_saber11','estrato','nivel_educ_padre']
    for v in [x for x in vars_cont if x in df.columns]:
        x0 = d0[v].dropna()
        x1 = d1[v].dropna()
        tt = stats.ttest_ind(x0, x1, equal_var=False)
        mw = stats.mannwhitneyu(x0, x1, alternative='two-sided')
        rows.append({
            'variable' : v,
            'tipo'     : 'continua',
            'media_p0' : round(x0.mean(),2),
            'de_p0'    : round(x0.std(),2),
            'n_p0'     : len(x0),
            'media_p1' : round(x1.mean(),2),
            'de_p1'    : round(x1.std(),2),
            'n_p1'     : len(x1),
            'delta'    : round(x1.mean()-x0.mean(),2),
            't_stat'   : round(tt.statistic,3),
            'p_ttest'  : round(tt.pvalue,4),
            'p_mw'     : round(mw.pvalue,4),
        })

    # Variables dicotómicas
    vars_bin = ['genero','estu_trabaja','internet','area_residencia',
                'naturaleza_ies','estu_cabeza_familia']
    for v in [x for x in vars_bin if x in df.columns]:
        x0 = d0[v].dropna()
        x1 = d1[v].dropna()
        p0, p1 = x0.mean(), x1.mean()
        tab = pd.crosstab(df[v], df['periodo_ia'])
        chi2_stat, chi2_p, _, _ = stats.chi2_contingency(tab, correction=True)
        rows.append({
            'variable' : v,
            'tipo'     : 'dicotomica (% = 1)',
            'media_p0' : round(p0*100,1),
            'de_p0'    : np.nan,
            'n_p0'     : len(x0),
            'media_p1' : round(p1*100,1),
            'de_p1'    : np.nan,
            'n_p1'     : len(x1),
            'delta'    : round((p1-p0)*100,1),
            't_stat'   : round(chi2_stat,3),
            'p_ttest'  : round(chi2_p,4),
            'p_mw'     : np.nan,
        })

    tabla3 = pd.DataFrame(rows)
    tabla3['etiqueta'] = tabla3['variable'].map(ETIQUETAS_VARS).fillna(tabla3['variable'])
    return tabla3

tabla3 = comparar_periodos(df)
guardar_tabla(tabla3, 'T3_descriptivo_periodos', TBL_DIR)
tabla3[['etiqueta','media_p0','de_p0','n_p0','media_p1','de_p1','n_p1','delta','p_ttest']]

In [ ]:
# ── Prueba de diferencia de medias en puntajes (con IC 95%) ───────────────
def prueba_diferencia_puntajes(df):
    vars_p = ['puntaje_saberpro_generico'] + MODULOS_GENERICOS
    rows = []
    for v in [x for x in vars_p if x in df.columns]:
        x0 = df[df['periodo_ia']==0][v].dropna()
        x1 = df[df['periodo_ia']==1][v].dropna()
        tt = stats.ttest_ind(x0, x1, equal_var=False)
        mw = stats.mannwhitneyu(x0, x1, alternative='two-sided')
        # IC 95% para la diferencia
        se_d = np.sqrt(x0.var()/len(x0) + x1.var()/len(x1))
        ic_l = (x1.mean()-x0.mean()) - 1.96*se_d
        ic_u = (x1.mean()-x0.mean()) + 1.96*se_d
        rows.append({
            'modulo'      : ETIQUETAS_VARS.get(v, v),
            'media_2122'  : round(x0.mean(),2),
            'media_2324'  : round(x1.mean(),2),
            'diferencia'  : round(x1.mean()-x0.mean(),2),
            'ic_95_inf'   : round(ic_l,2),
            'ic_95_sup'   : round(ic_u,2),
            't_stat'      : round(tt.statistic,3),
            'gl'          : round(tt.df,1),
            'p_ttest'     : round(tt.pvalue,4),
            'W_stat'      : round(mw.statistic,0),
            'p_mw'        : round(mw.pvalue,4),
        })
    tbl = pd.DataFrame(rows)
    # Corrección por pruebas múltiples
    reject_h, p_holm, _, _ = multipletests(tbl['p_ttest'], method='holm')
    reject_b, p_bh,   _, _ = multipletests(tbl['p_ttest'], method='fdr_bh')
    tbl['p_holm'] = p_holm.round(4)
    tbl['p_bh']   = p_bh.round(4)
    tbl['sig_holm'] = reject_h
    tbl['sig_bh']   = reject_b
    return tbl

tbl_puntajes = prueba_diferencia_puntajes(df)
guardar_tabla(tbl_puntajes, 'T3b_prueba_puntajes', TBL_DIR)
tbl_puntajes[['modulo','media_2122','media_2324','diferencia',
              'ic_95_inf','ic_95_sup','p_ttest','p_holm','sig_holm']]

In [ ]:
# ── Descriptivo por departamento ──────────────────────────────────────────
vars_p = ['puntaje_saberpro_generico'] + [m for m in MODULOS_GENERICOS if m in df.columns]
tbl_depto = (df.groupby(['depto_ies','periodo_ia'])[vars_p]
               .agg(['mean','std','count'])
               .round(2))
tbl_depto.columns = ['_'.join(c) for c in tbl_depto.columns]
tbl_depto = tbl_depto.reset_index()
guardar_tabla(tbl_depto, 'T_descriptivo_departamento', TBL_DIR)
print("Tabla por departamento:")
tbl_depto[['depto_ies','periodo_ia',
           'puntaje_saberpro_generico_mean','puntaje_saberpro_generico_std',
           'puntaje_saberpro_generico_count']]

In [ ]:
# ── Frecuencias de variables categóricas ──────────────────────────────────
for v in ['periodo_ia','depto_ies','programa_academico','genero','internet']:
    if v in df.columns:
        frq = df[v].value_counts(dropna=False).rename('n').reset_index()
        frq['pct'] = (frq['n'] / frq['n'].sum() * 100).round(1)
        print(f"\n--- {ETIQUETAS_VARS.get(v,v)} ---")
        print(frq.to_string(index=False))